In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import joblib
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#Load Data

In [1]:
!wget https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv

--2026-05-23 12:42:37--  https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66212309 (63M) [text/plain]
Saving to: ‘IMDB-Dataset.csv’

IMDB-Dataset.csv    100%[===================>]  63.14M   301MB/s    in 0.2s    

2026-05-23 12:42:39 (301 MB/s) - ‘IMDB-Dataset.csv’ saved [66212309/66212309]



In [2]:


df = pd.read_csv("IMDB-Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


#مرحله 2 — Text Cleaning


In [ ]:
'''

lowercase
حذف punctuation
حذف HTML
حذف stopwords
'''



def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text


df["review"] = df["review"].apply(clean_text)



#Tokenization

In [16]:
df["tokens"] = df["review"].apply(word_tokenize)
df.head()

,review,sentiment,tokens
0,One of the other reviewers has mentioned that ...,positive,"[One, of, the, other, reviewers, has, mentione..."
1,A wonderful little production. <br /><br />The...,positive,"[A, wonderful, little, production, ., <, br, /..."
2,I thought this was a wonderful way to spend ti...,positive,"[I, thought, this, was, a, wonderful, way, to,..."
3,Basically there's a family where a little boy ...,negative,"[Basically, there, 's, a, family, where, a, li..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"[Petter, Mattei, 's, ``, Love, in, the, Time, ..."


#Feature Extraction TF-IDF

In [17]:


vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["review"])

#Label Encoding

In [27]:
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

#Train/Test Split

In [19]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    df["sentiment"],
    test_size=0.2,
    random_state=42
)

#Train Model-Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

#PredictionPrediction

In [23]:
predictions = model.predict(X_test)

#Evaluation

In [24]:
accuracy = accuracy_score(y_test, predictions)

print(accuracy)

0.8954


#save

In [ ]:
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(model, 'sentiment_model.pkl')

In [ ]:
# بارگذاری
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('tfidf_vectorizer.pkl')


#######

In [28]:
import pandas as pd

df2 = pd.read_csv("IMDB-Dataset.csv")

df2.head()
X = df2['review']
y = df2['sentiment']

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression())
])

pipeline.fit(X, y)

# ذخیره کامل پایپ‌لاین
joblib.dump(pipeline, 'imdb_pipeline.pkl')

['imdb_pipeline.pkl']

In [30]:
pipeline = joblib.load('imdb_pipeline.pkl')

text = ["This movie was fantastic!"]

prediction = pipeline.predict(text)

print(prediction)

['positive']
